In [1]:
import jax
import jax.numpy as jnp
from flax import linen as nn
import optax
from jax import random
import tensorflow_datasets as tfds
from tqdm import tqdm
import time

# TCL2 Class in JAX
class TCL2(nn.Module):
    input_shape: tuple
    rank: tuple
    bias: bool = False

    def setup(self):
        self.w1 = self.param('w1', nn.initializers.xavier_uniform(), (self.input_shape[0], self.rank[0]))
        self.w2 = self.param('w2', nn.initializers.xavier_uniform(), (self.input_shape[1], self.rank[1]))
        self.w3 = self.param('w3', nn.initializers.xavier_uniform(), (self.input_shape[2], self.rank[2]))

    def __call__(self, x):
        batch_size = x.shape[0]
        x = jnp.reshape(x, (batch_size, -1))
        kronecker_product = jnp.kron(jnp.kron(self.w1, self.w2), self.w3)
        return jnp.dot(x, kronecker_product)

# CNN3 Class in JAX
class CNN3(nn.Module):
    @nn.compact
    def __call__(self, x):
        x = nn.Conv(32, kernel_size=(3, 3), strides=(1, 1), padding='SAME')(x)
        x = nn.relu(x)
        x = nn.max_pool(x, window_shape=(2, 2), strides=(2, 2))
        x = nn.Conv(64, kernel_size=(3, 3), strides=(1, 1), padding='SAME')(x)
        x = nn.relu(x)
        x = nn.max_pool(x, window_shape=(2, 2), strides=(2, 2))
        
        # Pass through TCL layers
        x = TCL2(input_shape=(64, 8, 8), rank=(64, 2, 2))(x)
        x = jnp.squeeze(x)
        x = TCL2(input_shape=(64, 2, 2), rank=(10, 1, 1))(x)
        return x

# Load CIFAR-10 dataset
def preprocess(data):
    image, label = data['image'], data['label']
    image = jnp.array(image) / 255.0  # Normalize to [0, 1]
    return image, label

def create_dataloader(split, batch_size):
    dataset = tfds.load('cifar10', split=split, as_supervised=True)
    dataset = dataset.map(preprocess)
    dataset = dataset.batch(batch_size)
    return dataset

# Create train and test loaders
train_loader = create_dataloader('train', batch_size=128)
test_loader = create_dataloader('test', batch_size=128)

# Initialize the model and optimizer
key = random.PRNGKey(0)
model = CNN3()
params = model.init(key, jnp.ones((1, 32, 32, 3)))  # CIFAR-10 image shape

optimizer = optax.adam(learning_rate=1e-3)
opt_state = optimizer.init(params)

# Define loss function
def cross_entropy_loss(logits, labels):
    return -jnp.mean(jax.nn.log_softmax(logits) * jax.nn.one_hot(labels, num_classes=10))

# Training function
def train_epoch(train_loader, epoch, params, opt_state):
    total_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in tqdm(train_loader):
        inputs = jnp.array(inputs)  # Convert to JAX array
        targets = jnp.array(targets)

        def loss_fn(params):
            logits = model.apply(params, inputs)
            return cross_entropy_loss(logits, targets)

        grad = jax.grad(loss_fn)(params)
        updates, opt_state = optimizer.update(grad, opt_state)
        params = optax.apply_updates(params, updates)

        loss = loss_fn(params)
        total_loss += loss
        correct += jnp.sum(jnp.argmax(logits, axis=-1) == targets)
        total += targets.shape[0]

    avg_loss = total_loss / len(train_loader)
    accuracy = correct / total
    print(f"Epoch {epoch}, Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}")

    return params, opt_state

# Test function
def test_epoch(test_loader, params):
    total_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in test_loader:
        inputs = jnp.array(inputs)
        targets = jnp.array(targets)

        logits = model.apply(params, inputs)
        loss = cross_entropy_loss(logits, targets)
        total_loss += loss
        correct += jnp.sum(jnp.argmax(logits, axis=-1) == targets)
        total += targets.shape[0]

    avg_loss = total_loss / len(test_loader)
    accuracy = correct / total
    print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.4f}")

# Training loop
n_epoch = 30
total_training_time = 0
total_testing_time = 0

st = time.time()
for epoch in range(1, n_epoch + 1):
    start_time = time.time()
    params, opt_state = train_epoch(train_loader, epoch, params, opt_state)
    train_time = time.time() - start_time

    start_time = time.time()
    test_epoch(test_loader, params)
    test_time = time.time() - start_time

    total_training_time += train_time
    total_testing_time += test_time

    print(f"Training time for epoch {epoch}: {train_time:.2f}s")
    print(f"Testing time for epoch {epoch}: {test_time:.2f}s")

elapsed = time.time() - st

# Total elapsed time for all epochs
# print(f"Total training time for {n_epoch} epochs: {total_training_time:.2f}s")
# print(f"Total testing time for {n_epoch} epochs: {total_testing_time:.2f}s")

print(f'total elapsed time : {elapsed}')




***************************************************************
Failed to import TensorFlow. Please note that TensorFlow is not installed by default when you install TFDS. This allows you to choose to install either `tf-nightly` or `tensorflow`. Please install the most recent version of TensorFlow, by following instructions at https://tensorflow.org/install.
***************************************************************




ModuleNotFoundError: Failed to construct dataset "cifar10", builder_kwargs "{'data_dir': None}": No module named 'tensorflow'